# 03 - Leakoff-Regime Classification

# DFIT Pressure Diagnostics
Developed as part of DFIT pressure-diagnostics research in the Harold Vance Department of Petroleum Engineering at Texas A&M University.

Primary reference: Barree, Miskimins & Gilbert (2015), SPE-169539-PA.

The shape of G*dP/dG relative to the through-origin reference line diagnoses the leakoff mechanism. This notebook walks through all four canonical Barree signatures and shows the classifier recovering each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dfit
from dfit.derivatives import semilog_derivative

## The four signatures

1. **Normal** - straight line through the origin; closure is the departure below it.
2. **Pressure-dependent leakoff (PDL)** - a hump ABOVE the line, from secondary fractures opening at high net pressure.
3. **Fracture-height recession / transverse storage** - a belly BELOW the line, from variable fracture compliance.
4. **Fracture-tip extension** - the straight section has a positive intercept (does not pass through the origin).

In [ ]:
regimes = list(dfit.REGIMES)
fig, axes = plt.subplots(2, 2, figsize=(11,8))
for ax, regime in zip(axes.ravel(), regimes):
    d = dfit.generate_dfit(regime=regime, closure_G=6.0, noise_psi=0.0, seed=2)
    fall = np.isfinite(d.G); G = d.G[fall]; p = d.pressure_psi[fall]
    o = np.argsort(G); G, p = G[o], p[o]
    sgd = semilog_derivative(G, p, smooth=5)
    pre = (G>0)&(G<=6.0)
    Gp, sp = G[pre], sgd[pre]
    early = Gp <= 0.35*Gp.max()
    slope = np.sum(Gp[early]*sp[early])/np.sum(Gp[early]**2)
    ax.plot(Gp, sp, 'r-', lw=1.3, label='G*dP/dG')
    ax.plot(Gp, slope*Gp, 'k--', lw=1, label='reference')
    res = dfit.classify_leakoff(G, p, closure_G=6.0)
    ax.set_title(f'{regime}  ->  classified: {res.regime}', fontsize=10)
    ax.set_xlabel('G'); ax.set_ylabel('G*dP/dG (psi)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Confidence and shape metrics

The classifier reports a normalized signed area (positive = hump, negative = belly) and an intercept ratio (positive = tip extension), plus a confidence equal to the margin between the top two regime scores.

In [ ]:
for regime in dfit.REGIMES:
    d = dfit.generate_dfit(regime=regime, closure_G=6.0, seed=3)
    fall = np.isfinite(d.G); G = d.G[fall]; p = d.pressure_psi[fall]
    o = np.argsort(G); G, p = G[o], p[o]
    r = dfit.classify_leakoff(G, p, closure_G=6.0)
    print(f'{regime:20s} -> {r.regime:20s} '
          f'area={r.normalized_area:+.2f} icpt={r.intercept_ratio:+.2f} '
          f'conf={r.confidence:.2f}')

### Why this matters

Mis-reading the leakoff regime corrupts the closure pick, and through it the closure stress, net pressure, and permeability. Pressure-dependent leakoff in particular is routinely mistaken for normal leakoff, leading to a closure pick inside the hump and an over-estimated closure stress.